# TrustLine — Tier 2 SaaS on Google Colab

Colab acts as the **regional SaaS** (ap-south): Postgres (RLS), NATS JetStream, nginx mTLS ingest, control plane,
ingest-api, workers (health + AIOps) and the dashboard API. Your **laptop stays Tier 1**: simulator + collector.

```
laptop: simulator -> collector --mTLS 1.3--> bore.pub:PORT (raw TCP relay) --> Colab nginx:8443 -> ingest -> NATS -> workers -> Postgres
browser --> https://xxxx.trycloudflare.com --> Colab api:8000 (dashboard + REST + WebSocket)
```

The bore relay only forwards TCP, so TLS ends **inside Colab** — the relay sees ciphertext only.

**Before you start (on the laptop):**
1. `python scripts/tl.py init --local` (once)
2. Regenerate certs so the server cert also names `bore.pub`:
   `EXTRA_SAN=DNS:bore.pub FORCE=1 sh deploy/certs/gen.sh` (Git Bash)
3. `python scripts/native.py bundle` → upload `dist/trustline-colab.zip` below.

**Limits:** sessions end after ~12 h or ~90 min idle; data is lost when the runtime resets; keep this tab open.

In [ ]:
#@title 1. Upload the bundle (dist/trustline-colab.zip from your laptop)
import os, zipfile
from google.colab import files
os.chdir('/content')
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/trustline')
os.chdir('/content/trustline')
print('extracted', len(os.listdir('.')), 'entries; .env present:', os.path.exists('.env'),
      '| dashboard built:', os.path.exists('dashboard/dist/index.html'))

In [ ]:
#@title 2. Install (Postgres, nginx, NATS, tunnels, Python deps) - about 2 min
%cd /content/trustline
import sys
sys.path.insert(0, 'scripts')
import native
native.setup_linux(timescale=False)   # True tries TimescaleDB; plain Postgres works too
print('setup done')

In [ ]:
#@title 2b. (Optional) AI Investigator: use Gemini (or Claude) instead of the in-region deterministic engine
# The bundle's .env already carries GEMINI_API_KEY if you set it on the laptop. To override it, add a
# Colab Secret named GEMINI_API_KEY (or ANTHROPIC_API_KEY) via the key icon and enable notebook access.
# Without any key the investigator still works, using the deterministic engine (no data leaves Colab).
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None
found = []
for name in ('GEMINI_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        v = userdata.get(name) if userdata else None
    except Exception:
        v = None
    if v:
        os.environ[name] = v
        found.append(name)
env_file = '/content/trustline/.env'
in_env = os.path.exists(env_file) and any(l.startswith('GEMINI_API_KEY=') and l.strip() != 'GEMINI_API_KEY=' for l in open(env_file))
if found:
    print('AI Investigator: using Colab secret(s)', ', '.join(found))
elif in_env:
    print('AI Investigator: GEMINI_API_KEY from the bundle .env will be used')
else:
    print('AI Investigator: no key - deterministic engine will be used')


In [ ]:
#@title 3. Start the SaaS + tunnels (runs in the background)
import subprocess, time, re, pathlib
%cd /content/trustline
subprocess.run('killall -9 nginx nats-server bore cloudflared 2>/dev/null || true', shell=True)
subprocess.run('fuser -k 8443/tcp 8000/tcp 8001/tcp 8010/tcp 4222/tcp 2>/dev/null || true', shell=True)
subprocess.run('python scripts/native.py stop 2>/dev/null || true', shell=True)
subprocess.run('rm -f .run/logs/bore.log .run/logs/cloudflared.log /content/saas.log', shell=True)
subprocess.run('sed -i "s/listen 8443 ssl;/listen 8443 ssl http2;/" /content/trustline/deploy/nginx/templates/ingest.conf.template', shell=True)
subprocess.run('sed -i "/http2 on;/d" /content/trustline/deploy/nginx/templates/ingest.conf.template', shell=True)
time.sleep(1)
log = open('/content/saas.log', 'w')
proc = subprocess.Popen(['python', '-u', 'scripts/native.py', 'saas', '--tunnel'], stdout=log, stderr=subprocess.STDOUT)
bore_url, cf_url = None, None
for _ in range(60):
    time.sleep(1)
    if not bore_url and pathlib.Path('.run/logs/bore.log').exists():
        m = re.findall(r'listening at (bore\.pub:\d+)', pathlib.Path('.run/logs/bore.log').read_text(errors='replace'))
        if m: bore_url = m[-1]
    if not cf_url and pathlib.Path('.run/logs/cloudflared.log').exists():
        m = re.findall(r'(https://[a-z0-9-]+\.trycloudflare\.com)', pathlib.Path('.run/logs/cloudflared.log').read_text(errors='replace'))
        if m: cf_url = m[-1]
    if bore_url and cf_url:
        break
print('=' * 65)
print(f'LIVE DASHBOARD:  {cf_url}')
print(f'LIVE INGEST:     https://{bore_url}')
print('=' * 65)


In [ ]:
#@title 4. Status / logs
!tail -n 5 /content/trustline/.run/logs/{ingest,workers,api}.log
!curl -s localhost:8000/api/healthz; echo
!su postgres -c "psql -d trustline -tAc \"select count(*) as payloads from received_payloads\""

In [ ]:
#@title 5. Keep-alive (optional): prints a heartbeat every minute so the runtime looks busy
import time, urllib.request, json
for i in range(600):
    try:
        h = json.load(urllib.request.urlopen('http://127.0.0.1:8000/api/healthz', timeout=3))
        print(time.strftime('%H:%M:%S'), h)
    except Exception as e:
        print('api not answering:', e)
    time.sleep(60)

In [ ]:
#@title 6. Stop
!python scripts/native.py stop; nginx -c /content/trustline/.run/nginx.conf -p /content/trustline/.run -s stop